# E3 · Primer ensayo de desarrollo · Resultados en GitHub

**CNN-LSTM causal con atención penalizada por discordancia y saturación radial**, frente al control `M00` sin ambas modificaciones. Es una propuesta exploratoria; su elección definitiva, novedad y superioridad permanecen abiertas.

1. Selecciona una GPU en **Entorno de ejecución → Cambiar tipo de entorno**. El código verifica la GPU realmente asignada.
2. Ejecuta las celdas en orden. Se reutiliza el entorno aislado si ya está preparado en esta misma sesión.
3. Colab lee el secreto **`GITHUB_TOKEN`**, habilitado para este cuaderno, con permiso de **Contents: Read and write** en `jassan-bit/DeepLearning_202630`. Si falta, aparece un campo oculto para introducirlo. La credencial se mantiene en memoria.
4. Los resultados se publican en la rama **`e3-desarrollo-resultados-v1`**, carpeta **`runs/desarrollo_v1`**, del mismo repositorio. Las reconexiones recuperan allí los pesos y el saldo consumido.
5. La celda **Entrenar y guardar en GitHub** ejecuta una semilla, hasta 40 épocas por modelo y un límite acumulado de **1.200 segundos de ejecución del entrenador**. La instalación y los commits añaden tiempo de pared. Cada bloque normalmente contiene una época de cada modelo.

La validación guía la parada temprana. Estos resultados no prueban superioridad frente al Top 10 ni sustituyen la evaluación final de E3.

In [ ]:
from pathlib import Path
import hashlib
import json
import os
import subprocess
import sys
import urllib.request

REPOSITORY = "jassan-bit/DeepLearning_202630"
SOURCE_COMMIT = "aba0e5172ed2b22c4a0228fd7d0ba466c72acbc2"
RESULTS_BRANCH = "e3-desarrollo-resultados-v1"
RESULTS_URL = f"https://github.com/{REPOSITORY}/tree/{RESULTS_BRANCH}/runs/desarrollo_v1"
DEVICE = "cuda"
LOCAL_WORKSPACE = Path("/content/entregable3_development_v1")
SOURCE_DIR = LOCAL_WORKSPACE / ("source-" + SOURCE_COMMIT)
# Si la versión anterior dejó una corrida visible, conservarla al actualizar esta sesión.
PREVIOUS_RUN_DIR = globals().get("RUN_DIR")
RUN_DIR = LOCAL_WORKSPACE / "github-run-desarrollo-v1"
print("Código fijado:", SOURCE_COMMIT)
print("Resultados en GitHub:", RESULTS_URL)

## Descargar la versión fijada del código

Se descargan los archivos necesarios desde un commit inmutable. Los datos y el entorno de Python usan el almacenamiento temporal de Colab; los archivos de la corrida se conservan en la rama de resultados de GitHub.

In [ ]:
SOURCE_FILES = [
    "src/__init__.py", "src/original_model.py", "src/e3_development.py",
    "scripts/e3_run_development.py", "scripts/e3_github_storage.py",
    "notebooks/run_e3_github_colab.py", "experiments/e3_desarrollo_v1/plan.json",
    "notebooks/preparar_e3_desarrollo_runtime.py", "requirements-e3-development-gpu.lock.txt",
]

def download_pinned(relative_path, commit, destination, expected_sha256=None):
    url = f"https://raw.githubusercontent.com/{REPOSITORY}/{commit}/{relative_path}"
    with urllib.request.urlopen(url, timeout=120) as response:
        payload = response.read()
    digest = hashlib.sha256(payload).hexdigest()
    if expected_sha256 is not None and digest != expected_sha256:
        raise RuntimeError("SHA-256 distinto para " + relative_path)
    destination = Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)
    if destination.exists():
        if destination.read_bytes() != payload:
            raise RuntimeError("Archivo local modificado; se conserva: " + str(destination))
    else:
        temporary = destination.with_suffix(destination.suffix + ".download")
        temporary.write_bytes(payload)
        os.replace(temporary, destination)
    return digest

source_hashes = {name: download_pinned(name, SOURCE_COMMIT, SOURCE_DIR / name)
                 for name in SOURCE_FILES}
import importlib.util

def load_module(name, path):
    existing = sys.modules.get(name)
    if existing is not None and Path(existing.__file__).resolve() == Path(path).resolve():
        return existing
    spec = importlib.util.spec_from_file_location(name, path)
    module = importlib.util.module_from_spec(spec)
    sys.modules[name] = module
    spec.loader.exec_module(module)
    return module

GITHUB_HELPER = load_module("e3_github_colab", SOURCE_DIR / "notebooks/run_e3_github_colab.py")

## Autenticación de GitHub

Se utiliza el secreto `GITHUB_TOKEN` de Colab o, si falta, un campo oculto. El proceso que entrena los modelos recibe un entorno sin la credencial. Los commits incluyen únicamente los archivos admitidos de la corrida.

In [ ]:
GITHUB_AUTH_TOKEN = os.environ.get("GITHUB_TOKEN", "").strip()
if not GITHUB_AUTH_TOKEN:
    try:
        from google.colab import userdata
        GITHUB_AUTH_TOKEN = (userdata.get("GITHUB_TOKEN") or "").strip()
    except Exception:
        GITHUB_AUTH_TOKEN = ""
if not GITHUB_AUTH_TOKEN:
    from getpass import getpass
    GITHUB_AUTH_TOKEN = getpass("Token de GitHub con Contents: Read and write para este repositorio: ").strip()
if not GITHUB_AUTH_TOKEN or any(character.isspace() for character in GITHUB_AUTH_TOKEN):
    raise RuntimeError("Configura un GITHUB_TOKEN válido y vuelve a ejecutar esta celda.")
print("Credencial recibida en memoria. Se verificará el acceso al preparar la corrida.")

## Verificar el entorno y los datos

El entorno aislado usa Python 3.12.12, PyTorch 2.8.0 con CUDA 12.6 y NumPy 2.3.5. Si ya se instaló en esta sesión, se comprueba y reutiliza.

El NPZ está fijado por SHA-256. El entrenador sólo lee los tres arreglos originales de entrenamiento: **532 ejemplos de ajuste**, siete días purgados y **116 de validación interna**. Recalcula el preprocesamiento exclusivamente en ajuste.

In [ ]:
runtime_helper = load_module("e3_isolated_runtime", SOURCE_DIR / "notebooks/preparar_e3_desarrollo_runtime.py")
GPU_PYTHON, RUN_ENV, GPU_RUNTIME = runtime_helper.preparar_entorno(
    LOCAL_WORKSPACE / "runtime", device=DEVICE)
PLAN_PATH = SOURCE_DIR / "experiments/e3_desarrollo_v1/plan.json"
PLAN = json.loads(PLAN_PATH.read_text())
data_source = PLAN["data_source"]
DATA_PATH = LOCAL_WORKSPACE / "datos_auditados_v1" / "current_cohort_arrays.npz"
DATA_SHA256 = data_source["sha256"]
if DATA_PATH.exists():
    if hashlib.sha256(DATA_PATH.read_bytes()).hexdigest() != DATA_SHA256:
        raise RuntimeError("El archivo local de datos no coincide con la auditoría; se conserva.")
else:
    download_pinned(data_source["path"], data_source["commit"], DATA_PATH, DATA_SHA256)
print("Datos verificados:", DATA_SHA256)

## Recuperar o preparar y publicar la corrida

Esta celda recupera la corrida de GitHub o prepara sus pesos iniciales y publica el estado antes de entrenar. Un error de acceso o de sincronización detiene el avance. La recuperación conserva el registro de cómputo; no inicia otra corrida para sustituirlo.

Una reserva remota activa impide que otra sesión entrene simultáneamente. Tras un cierre abrupto, se conserva su fecha original y se aplica el cargo conservador correspondiente al recuperar; puede agotarse el saldo. Usa una sola sesión por corrida.

In [ ]:
# Importar una corrida anterior que siga visible en esta sesión, sin modificarla.
if PREVIOUS_RUN_DIR is not None:
    previous = Path(PREVIOUS_RUN_DIR)
    if previous != RUN_DIR and (previous / "manifest.json").is_file() and not RUN_DIR.exists():
        from scripts.e3_github_storage import build_snapshot, restore_snapshot
        import fcntl
        with (previous / ".run.lock").open("a+") as previous_lock:
            fcntl.flock(previous_lock, fcntl.LOCK_EX | fcntl.LOCK_NB)
            previous_ledger = json.loads((previous / "ledger.json").read_text())
            if previous_ledger.get("active_session"):
                raise RuntimeError("La corrida anterior sigue marcada activa. Se conserva para recuperar su estado; no se reinicia ni se importa automáticamente.")
            previous_manifest, previous_files = build_snapshot(previous)
            restore_snapshot(previous_manifest, previous_files, RUN_DIR)
        print("Se conservó la corrida anterior con sus pesos y su saldo de cómputo.")

GITHUB_ARGS = dict(token=GITHUB_AUTH_TOKEN, npz_path=DATA_PATH, expected_sha256=DATA_SHA256,
                   run_dir=RUN_DIR, plan_path=PLAN_PATH, code_root=SOURCE_DIR, device=DEVICE,
                   python=GPU_PYTHON, environment=RUN_ENV)
preparation = GITHUB_HELPER.prepare_github_run(**GITHUB_ARGS)
print(json.dumps(preparation, ensure_ascii=False, indent=2))
admission = json.loads((RUN_DIR / "data_admission.json").read_text())
assert admission["n_fit"] == data_source["expected_fit_count"]
assert admission["n_validation"] == data_source["expected_validation_count"]
assert admission["n_purged_or_unused"] == data_source["expected_purged_count"]
print("Ajuste / purga / validación:", admission["n_fit"], admission["n_purged_or_unused"], admission["n_validation"])
print("Archivos publicados:", RESULTS_URL)

## Entrenar y guardar en GitHub

Antes de cada bloque se confirma en GitHub su reserva activa. Al terminar, se publica el estado cerrado con pesos, optimizador, curvas, predicciones y saldo; sólo entonces puede comenzar el siguiente bloque. Cada época muestra su avance.

El límite de 1.200 segundos conserva el tiempo ya consumido. La instalación y los commits pueden añadir varios minutos. Si detienes esta celda, el ejecutor interrumpe el proceso de entrenamiento e intenta publicar el último estado coherente. Una publicación sin confirmar conserva los archivos locales y detiene el avance.

In [ ]:
execution = GITHUB_HELPER.run_github_development(**GITHUB_ARGS)
print(json.dumps(execution, ensure_ascii=False, indent=2))
print("Resultados y puntos de reanudación:", RESULTS_URL)

## Examinar el resultado de desarrollo

RMSE y MAE están en las unidades originales de σ7. Cada modelo se resume en su mejor época según validación; esa selección hace que estas métricas sean exploratorias. La pérdida de ajuste es la media de pérdidas por lote durante las actualizaciones, no una evaluación del checkpoint final. `M00` tiene un parámetro menos y retira dos componentes a la vez.

In [ ]:
from html import escape
from IPython.display import HTML, display

summary_path = RUN_DIR / "summary.json"
if not summary_path.is_file():
    print("Aún no hay resumen: revisa la salida del entrenamiento y los archivos de la corrida.")
else:
    summary = json.loads(summary_path.read_text())
    print("Estado:", summary["status"])
    print(f"Tiempo registrado: {summary['charged_seconds']:.1f} / {summary['limit_seconds']:.0f} s")
    rows = []
    for name, result in summary["models"].items():
        values = [name, result["parameters"], result["completed_epochs"], result["best_epoch"],
                  result["best_validation_rmse"], result["best_validation_mae"]]
        rows.append("<tr>" + "".join("<td>" + escape(str(value)) + "</td>" for value in values) + "</tr>")
    display(HTML("<table><thead><tr><th>Modelo</th><th>Parámetros</th><th>Épocas completas</th>"
                 "<th>Mejor época</th><th>RMSE validación interna</th><th>MAE validación interna</th>"
                 "</tr></thead><tbody>" + "".join(rows) + "</tbody></table>"))
    print("Archivos conservados en GitHub:", RESULTS_URL)
    print("summary.json, plan.json, manifest.json, ledger.json, history.jsonl, data_admission.json")
    print("Cada modelo: curvas, predicciones de validación, mejores pesos y punto de reanudación.")

In [ ]:
import csv
try:
    import plotly.graph_objects as go
except ImportError:
    print("Plotly no está disponible en el kernel; las curvas están en los CSV de GitHub.")
else:
    figure = go.Figure()
    for name in ("full", "M00"):
        curves_path = RUN_DIR / name / "curves.csv"
        if curves_path.is_file():
            with curves_path.open() as stream:
                rows = list(csv.DictReader(stream))
            figure.add_trace(go.Scatter(x=[int(row["epoch"]) for row in rows],
                                       y=[float(row["validation_RMSE_original"]) for row in rows],
                                       mode="lines+markers", name=name))
    figure.update_layout(title="Validación interna de desarrollo · una semilla",
                         xaxis_title="Época", yaxis_title="RMSE de σ7 (unidades originales)",
                         template="plotly_white")
    figure.show()

## Siguiente decisión

Revisar las curvas, el coste, los errores y la comparación contra `M00`. Una mejora permite continuar estudiando el mecanismo. Antes de una campaña final siguen pendientes la revisión de novedad, la comprensión del código por el autor, las ablaciones individuales, el control de capacidad, el análisis de potencia y el protocolo pre-registrado. La declaración de asistencia de IA debe incluir este trabajo.